# Shot-Count Convergence Analysis — N_SHOTS = 100,000

Runs both circuit topologies at 100k shots, computes Wilson score CIs,
saves histogram PDFs, and prints the top-5 table + Spearman ρ for Supplementary Table S1.

**Outputs** (written to `figures/`):
- `100000_shot_histogram_co.pdf` → Supplementary Fig. S1
- `100000_shot_histogram_mo.pdf` → Supplementary Fig. S2
- Console output → fill Supplementary Table S1

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — safe for saving PDFs
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import os, pathlib

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qsim_cells.generative import (
    create_rotation_circuit,
    concatenate_circuits_with_separate_measurements,
    add_crx_and_measurements_to_circuit,
)

# ── Parameters (must match main notebook) ────────────────────────────────────
MY_SEED  = 42
N_SHOTS  = 100_000
N_SHOTS_2K = 2_000   # reference run for convergence comparison

ANG_CT1 = np.array([0.2, 0.1, 0.4, 0.9, 0.8]) * np.pi
ANG_CT2 = np.array([0.2, 0.3, 0.2, 0.7, 0.5]) * np.pi

INTERACTION_CASE1 = [(3, 5), (5, 7), (7, 0)]   # L1 — inter-state cascade
INTERACTION_CASE2 = [(2, 1)]                     # L2 — non-interacting control

FIG_DIR = pathlib.Path('figures')
FIG_DIR.mkdir(exist_ok=True)

np.random.seed(MY_SEED)
print(f'N_SHOTS={N_SHOTS}  seed={MY_SEED}  fig_dir={FIG_DIR.resolve()}')

N_SHOTS=100000  seed=42  fig_dir=/home/ssromerogon/github_repos/qSimCells/figures


In [8]:
# ── Helper: run circuit and return raw counts dict ───────────────────────────

def run_circuit(ang_ct1, ang_ct2, interaction_map, n_shots, seed, backend=None):
    """Returns (counts_ct1, counts_ct2) as dicts {bitstring: count}."""
    np.random.seed(seed)
        
    circ1    = create_rotation_circuit(ang_ct1)
    circ2    = create_rotation_circuit(ang_ct2)
    combined = concatenate_circuits_with_separate_measurements(circ1, circ2)
    final    = add_crx_and_measurements_to_circuit(combined, circ1.num_qubits, interaction_map)

    if backend is None:
        backend = AerSimulator(seed_simulator=seed)

    try:
        pm      = generate_preset_pass_manager(backend=backend, optimization_level=3)
        qc_comp = pm.run(final)
    except Exception:
        qc_comp = final
    
    result = Sampler(mode=backend).run([qc_comp], shots=n_shots).result()[0]

    reg_names       = [cr.name for cr in final.cregs]
    counts_ct1 = result.data.c_measure1.get_counts() if 'c_measure1' in reg_names else None
    counts_ct2 = result.data.c_measure2.get_counts() if 'c_measure2' in reg_names else None

    return counts_ct1, counts_ct2


def counts_to_freq(counts_dict, n_qubits):
    """Convert counts dict → numpy array of length 2**n_qubits (indexed by decimal state)."""
    n_states = 2 ** n_qubits
    total    = sum(counts_dict.values())
    freq     = np.zeros(n_states)
    for bitstr, cnt in counts_dict.items():
        idx = int(bitstr[::-1], 2)   # reverse for little-endian
        freq[idx] = cnt / total
    return freq


def wilson_ci(p, n, z=1.96):
    """Wilson score CI half-width for a proportion p estimated from n trials."""
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2*n)) / denom
    half   = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return centre, half

print('Helpers defined.')

Helpers defined.


In [9]:
# ── Run both topologies at 2k and 100k ───────────────────────────────────────
print('Running L1 (inter-state cascade) at 2k shots...')
c1_2k_ct1, c1_2k_ct2 = run_circuit(ANG_CT1, ANG_CT2, INTERACTION_CASE1, N_SHOTS_2K, MY_SEED)

print('Running L1 at 100k shots...')
c1_100k_ct1, c1_100k_ct2 = run_circuit(ANG_CT1, ANG_CT2, INTERACTION_CASE1, N_SHOTS, MY_SEED)

print('Running L2 (non-interacting control) at 100k shots...')
c2_100k_ct1, c2_100k_ct2 = run_circuit(ANG_CT1, ANG_CT2, INTERACTION_CASE2, N_SHOTS, MY_SEED)

print('All circuits done.')

Running L1 (inter-state cascade) at 2k shots...
Running L1 at 100k shots...
Running L2 (non-interacting control) at 100k shots...
All circuits done.


In [10]:
# ── Plot histogram with Wilson CI error bars and save as PDF ─────────────────

def plot_histogram_with_ci(counts_ct1, counts_ct2, n_shots, title, save_path):
    n_qubits = 5
    n_states = 2 ** n_qubits
    states   = np.arange(n_states)

    freq1 = counts_to_freq(counts_ct1, n_qubits)
    freq2 = counts_to_freq(counts_ct2, n_qubits)

    _, ci1 = zip(*[wilson_ci(p, n_shots) for p in freq1])
    _, ci2 = zip(*[wilson_ci(p, n_shots) for p in freq2])
    ci1, ci2 = np.array(ci1), np.array(ci2)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, freq, ci, reg_label, color in zip(
            axes, [freq1, freq2], [ci1, ci2],
            ['CT1 (5 qubits)', 'CT2 (5 qubits)'],
            ['#2E86AB', '#E84855']):
        ax.bar(states, freq, color=color, alpha=0.75, label='Frequency')
        ax.errorbar(states, freq, yerr=ci, fmt='none',
                    ecolor='black', elinewidth=0.8, capsize=2)
        ax.set_xlabel('Basis state (decimal index)', fontsize=11)
        ax.set_ylabel('Probability', fontsize=11)
        ax.set_title(f'{reg_label}', fontsize=12)
        ax.set_xticks(states[::2])
        ax.tick_params(axis='x', labelsize=8)

    fig.suptitle(f'{title}  ($N_{{\\mathrm{{shots}}}}={n_shots:,}$, seed={MY_SEED})\n'
                 f'Error bars: 95% Wilson CI', fontsize=12)
    plt.tight_layout()
    fig.savefig(save_path, format='pdf', bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f'Saved: {save_path}')


plot_histogram_with_ci(
    c1_100k_ct1, c1_100k_ct2, N_SHOTS,
    title='Co-culture — inter-state cascade $L_1$',
    save_path=FIG_DIR / '100000_shot_histogram_co.pdf'
)

plot_histogram_with_ci(
    c2_100k_ct1, c2_100k_ct2, N_SHOTS,
    title='Mono-culture — non-interacting control $L_2$',
    save_path=FIG_DIR / '100000_shot_histogram_mo.pdf'
)

Saved: figures/100000_shot_histogram_co.pdf
Saved: figures/100000_shot_histogram_mo.pdf


In [12]:
# ── Convergence table: top-5 states + Spearman ρ ────────────────────────────
# Uses CT1 register of L1 (co-culture) — 5 qubits, 32 states

n_qubits = 5
N2K  = N_SHOTS_2K
N100 = N_SHOTS

freq_2k  = counts_to_freq(c1_2k_ct1,  n_qubits)
freq_100 = counts_to_freq(c1_100k_ct1, n_qubits)

# Raw counts at 2k
counts_2k_arr = np.zeros(2**n_qubits)
for bitstr, cnt in c1_2k_ct1.items():
    idx = int(bitstr[::-1], 2)
    counts_2k_arr[idx] = cnt

# Spearman over all 32 states
rho, pval = spearmanr(freq_2k, freq_100)
print(f'Spearman ρ (all 32 states, 2k vs 100k): {rho:.4f}  (p={pval:.2e})')
print()

# Wilson CI at 100k
_, ci_100 = zip(*[wilson_ci(p, N100) for p in freq_100])
ci_100 = np.array(ci_100)

# Top-5 by 100k frequency
top5_idx = np.argsort(freq_100)[::-1][:5]

print('─' * 72)
print(f'{"State":>6}  {"Count(2k)":>10}  {"Freq(2k,%)": >11}  {"Freq(100k,%)": >13}  {"95%CI(±%)": >11}')
print('─' * 72)
for idx in top5_idx:
    print(f'{idx:>6}  {int(counts_2k_arr[idx]):>10}  '
          f'{freq_2k[idx]*100:>10.2f}%  '
          f'{freq_100[idx]*100:>12.2f}%  '
          f'{ci_100[idx]*100:>10.3f}%')
print('─' * 72)
print(f'Spearman ρ = {rho:.4f}')
print()

Spearman ρ (all 32 states, 2k vs 100k): 0.9324  (p=8.47e-15)

────────────────────────────────────────────────────────────────────────
 State   Count(2k)   Freq(2k,%)   Freq(100k,%)    95%CI(±%)
────────────────────────────────────────────────────────────────────────
    19         844       42.20%         42.88%       0.307%
    23         456       22.80%         22.72%       0.260%
     3         263       13.15%         13.20%       0.210%
     7         154        7.70%          7.06%       0.159%
    18          92        4.60%          4.63%       0.130%
────────────────────────────────────────────────────────────────────────
Spearman ρ = 0.9324

